# Hệ thống dự báo chuỗi thời gian: LSTM vs phương pháp truyền thống

Notebook so sánh **Linear Regression**, **Random Forest**, **ARIMA** và **LSTM** trên dữ liệu tài chính (ACB) và môi trường (Air Quality).

## 1. Introduction

- Khám phá và trực quan hóa riêng hai bộ dữ liệu trước khi huấn luyện.
- Tiền xử lý: missing value, chuẩn hóa, tạo chuỗi thời gian (lag features).
- Huấn luyện và đánh giá RMSE, MAE, R²; LSTM thêm Training Time và Prediction Time.
- So sánh mô hình, tối ưu độ trễ, ablation study và đánh giá dung lượng lưu trữ.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from statsmodels.tsa.arima.model import ARIMA

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded')


## 2. Dataset Exploration

Mục tiêu: hiểu cấu trúc, phân phối và xu hướng của dữ liệu trước khi huấn luyện.

### 2.1 ACB Dataset

In [ ]:
acb = pd.read_csv('ACB.csv')
acb['timestamp'] = pd.to_datetime(acb['timestamp'])
acb = acb.sort_values('timestamp').reset_index(drop=True)
acb.head()


In [ ]:
print('ACB shape:', acb.shape)
print(acb.dtypes)
print('\nMissing values:')
print(acb.isna().sum())
acb.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(acb['timestamp'], acb['close'], color='steelblue')
axes[0].set_title('ACB - Close Price Trend')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Close')
sns.histplot(acb['close'], kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('ACB - Close Price Distribution')
plt.tight_layout()
plt.show()


### 2.2 AirQuality Dataset

In [ ]:
aq_path = Path('air+quality/AirQualityUCI.csv')
airquality = pd.read_csv(aq_path, sep=';', decimal='.', header=0)
airquality = airquality.iloc[:, :-2]
airquality.columns = [c.strip() for c in airquality.columns]
airquality['Date'] = pd.to_datetime(airquality['Date'], format='%d/%m/%Y', errors='coerce')
airquality['Time'] = pd.to_timedelta(
    airquality['Time'].astype(str).str.replace('.', ':', regex=False)
)
airquality['timestamp'] = airquality['Date'] + airquality['Time']
airquality = airquality.drop(columns=['Date', 'Time'])
numeric_cols = airquality.columns.drop('timestamp')
airquality[numeric_cols] = airquality[numeric_cols].apply(pd.to_numeric, errors='coerce')
airquality = airquality.sort_values('timestamp').reset_index(drop=True)

airquality_daily = (
    airquality[['timestamp', 'CO(GT)']]
    .set_index('timestamp')
    .resample('D')
    .mean()
    .reset_index()
)
airquality_daily.head()


In [ ]:
print('AirQuality (hourly) shape:', airquality.shape)
print('AirQuality (daily) shape:', airquality_daily.shape)
print('\nMissing CO(GT) daily:', airquality_daily['CO(GT)'].isna().sum())
airquality_daily.describe()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(airquality_daily['timestamp'], airquality_daily['CO(GT)'], color='seagreen')
axes[0].set_title('Air Quality - Daily CO(GT) Trend')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('CO (mg/m3)')
sns.histplot(airquality_daily['CO(GT)'].dropna(), kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Air Quality - CO(GT) Distribution')
plt.tight_layout()
plt.show()


## 3. Data Preprocessing

- Xử lý missing value
- Chuẩn hóa (StandardScaler)
- Tạo chuỗi thời gian bằng lag features và chia train/test theo thời gian (80/20)

In [ ]:
def create_lag_features(df, target_col, n_lags=5):
    lagged = pd.DataFrame({'timestamp': df['timestamp']})
    for lag in range(1, n_lags + 1):
        lagged[f'lag_{lag}'] = df[target_col].shift(lag)
    lagged['target'] = df[target_col]
    return lagged.dropna().reset_index(drop=True)


def time_series_split(X, y, test_size=0.2):
    split = int(len(X) * (1 - test_size))
    return X.iloc[:split], X.iloc[split:], y.iloc[:split], y.iloc[split:]


def evaluate_model(y_true, y_pred):
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)),
    }


def scale_features(X_train, X_test):
    scaler = StandardScaler()
    return scaler.fit_transform(X_train), scaler.transform(X_test), scaler


N_LAGS = 5
TEST_SIZE = 0.2


In [ ]:
# ACB preprocessing
acb_pre = acb[['timestamp', 'close']].copy()
acb_pre['close'] = acb_pre['close'].ffill().bfill()

acb_lag = create_lag_features(acb_pre, 'close', n_lags=N_LAGS)
X_acb = acb_lag[[c for c in acb_lag.columns if c.startswith('lag_')]]
y_acb = acb_lag['target']
X_acb_train, X_acb_test, y_acb_train, y_acb_test = time_series_split(X_acb, y_acb, TEST_SIZE)
X_acb_train_s, X_acb_test_s, acb_scaler = scale_features(X_acb_train, X_acb_test)
acb_series = acb_pre.set_index('timestamp')['close']

# Air Quality preprocessing
aq_pre = airquality_daily[['timestamp', 'CO(GT)']].copy()
aq_pre = aq_pre.rename(columns={'CO(GT)': 'value'})
aq_pre['value'] = aq_pre['value'].interpolate(method='linear').bfill().ffill()

aq_lag = create_lag_features(aq_pre, 'value', n_lags=N_LAGS)
X_aq = aq_lag[[c for c in aq_lag.columns if c.startswith('lag_')]]
y_aq = aq_lag['target']
X_aq_train, X_aq_test, y_aq_train, y_aq_test = time_series_split(X_aq, y_aq, TEST_SIZE)
X_aq_train_s, X_aq_test_s, aq_scaler = scale_features(X_aq_train, X_aq_test)
aq_series = aq_pre.set_index('timestamp')['value']

print('ACB train/test:', len(X_acb_train), len(X_acb_test))
print('AirQuality train/test:', len(X_aq_train), len(X_aq_test))


## 4. Linear Regression

Baseline đơn giản trên lag features đã chuẩn hóa.

In [ ]:
def run_lr(X_train, X_test, y_train, y_test):
    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return model, preds, evaluate_model(y_test, preds)


lr_acb, acb_lr_pred, acb_lr_metrics = run_lr(
    X_acb_train_s, X_acb_test_s, y_acb_train, y_acb_test
)
lr_aq, aq_lr_pred, aq_lr_metrics = run_lr(
    X_aq_train_s, X_aq_test_s, y_aq_train, y_aq_test
)

print('ACB Linear Regression:', acb_lr_metrics)
print('AirQuality Linear Regression:', aq_lr_metrics)


In [ ]:
test_ts_acb = acb_lag['timestamp'].iloc[-len(y_acb_test):]
plt.figure(figsize=(12, 4))
plt.plot(test_ts_acb, y_acb_test.values, label='Actual')
plt.plot(test_ts_acb, acb_lr_pred, label='LR Prediction')
plt.title('ACB - Linear Regression Forecast')
plt.legend()
plt.tight_layout()
plt.show()


## 5. Random Forest

Mô hình ensemble phi tuyến trên lag features gốc (không scale).

In [ ]:
def run_rf(X_train, X_test, y_train, y_test):
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return model, preds, evaluate_model(y_test, preds)


rf_acb, acb_rf_pred, acb_rf_metrics = run_rf(
    X_acb_train, X_acb_test, y_acb_train, y_acb_test
)
rf_aq, aq_rf_pred, aq_rf_metrics = run_rf(
    X_aq_train, X_aq_test, y_aq_train, y_aq_test
)

print('ACB Random Forest:', acb_rf_metrics)
print('AirQuality Random Forest:', aq_rf_metrics)


## 6. ARIMA

Tìm tham số (p, d, q) tối ưu theo AIC trên tập train.

In [ ]:
def select_arima_order(series, p_range=range(0, 3), d_range=range(0, 2), q_range=range(0, 3)):
    best_aic, best_order = np.inf, None
    for p in p_range:
        for d in d_range:
            for q in q_range:
                try:
                    res = ARIMA(series, order=(p, d, q)).fit()
                    if res.aic < best_aic:
                        best_aic, best_order = res.aic, (p, d, q)
                except Exception:
                    continue
    return best_order


def run_arima(train_series, test_len, y_test):
    order = select_arima_order(train_series)
    fitted = ARIMA(train_series, order=order).fit()
    preds = fitted.forecast(steps=test_len)
    metrics = evaluate_model(y_test.values, preds)
    return order, preds, metrics


acb_train_len = len(y_acb_train)
aq_train_len = len(y_aq_train)

acb_order, acb_arima_pred, acb_arima_metrics = run_arima(
    acb_series.iloc[:acb_train_len], len(y_acb_test), y_acb_test
)
aq_order, aq_arima_pred, aq_arima_metrics = run_arima(
    aq_series.iloc[:aq_train_len], len(y_aq_test), y_aq_test
)

print('ACB ARIMA order:', acb_order, acb_arima_metrics)
print('AirQuality ARIMA order:', aq_order, aq_arima_metrics)


## 7. LSTM

Huấn luyện LSTM trên sequences từ lag features; đo thời gian train và predict.

In [ ]:
def prepare_lstm_sequences(X_scaled, y, n_steps=5):
    Xs, ys = [], []
    y = np.asarray(y)
    for i in range(len(X_scaled) - n_steps + 1):
        Xs.append(X_scaled[i : i + n_steps])
        ys.append(y[i + n_steps - 1])
    return np.array(Xs), np.array(ys)


def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, activation='tanh', input_shape=input_shape),
        Dropout(0.2),
        Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


N_STEPS = 5
es = EarlyStopping(patience=10, restore_best_weights=True, monitor='val_loss')


In [ ]:
def run_lstm(X_train_s, y_train, X_test_s, y_test, epochs=50, batch_size=16):
    X_tr, y_tr = prepare_lstm_sequences(X_train_s, y_train, N_STEPS)
    X_concat = np.vstack([X_train_s[-(N_STEPS - 1) :], X_test_s])
    y_concat = np.concatenate([y_train.values[-(N_STEPS - 1) :], y_test.values])
    X_te, y_te = prepare_lstm_sequences(X_concat, y_concat, N_STEPS)

    model = build_lstm((X_tr.shape[1], X_tr.shape[2]))
    t0 = time.time()
    model.fit(
        X_tr,
        y_tr,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        callbacks=[es],
        verbose=0,
    )
    train_time = time.time() - t0

    t1 = time.time()
    preds = model.predict(X_te, verbose=0).flatten()
    predict_time = time.time() - t1

    metrics = evaluate_model(y_te, preds)
    metrics['Train Time (s)'] = train_time
    metrics['Predict Time (s)'] = predict_time
    return model, preds, y_te, metrics


acb_lstm, acb_lstm_pred, y_acb_lstm_test, acb_lstm_metrics = run_lstm(
    X_acb_train_s, y_acb_train, X_acb_test_s, y_acb_test
)
aq_lstm, aq_lstm_pred, y_aq_lstm_test, aq_lstm_metrics = run_lstm(
    X_aq_train_s, y_aq_train, X_aq_test_s, y_aq_test
)

print('ACB LSTM:', acb_lstm_metrics)
print('AirQuality LSTM:', aq_lstm_metrics)


## 8. Model Comparison

In [ ]:
def comparison_table(lr_m, rf_m, arima_m, lstm_m):
    return pd.DataFrame([
        {
            'Model': 'Linear Regression',
            'RMSE': lr_m['RMSE'],
            'MAE': lr_m['MAE'],
            'R2': lr_m['R2'],
        },
        {
            'Model': 'Random Forest',
            'RMSE': rf_m['RMSE'],
            'MAE': rf_m['MAE'],
            'R2': rf_m['R2'],
        },
        {
            'Model': 'ARIMA',
            'RMSE': arima_m['RMSE'],
            'MAE': arima_m['MAE'],
            'R2': arima_m['R2'],
        },
        {
            'Model': 'LSTM',
            'RMSE': lstm_m['RMSE'],
            'MAE': lstm_m['MAE'],
            'R2': lstm_m['R2'],
        },
    ]).set_index('Model')


comparison_acb = comparison_table(
    acb_lr_metrics, acb_rf_metrics, acb_arima_metrics, acb_lstm_metrics
)
comparison_aq = comparison_table(
    aq_lr_metrics, aq_rf_metrics, aq_arima_metrics, aq_lstm_metrics
)

print('=== ACB Dataset ===')
display(comparison_acb.round(4))
print('\n=== Air Quality Dataset ===')
display(comparison_aq.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in zip(
    axes, [comparison_acb, comparison_aq], ['ACB', 'Air Quality']
):
    df['RMSE'].plot(
        kind='bar',
        ax=ax,
        color=['#4C72B0', '#55A868', '#C44E52', '#8172B2'],
    )
    ax.set_title(f'{title} - RMSE by Model')
    ax.set_ylabel('RMSE')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


## 9. Latency Optimization

Đo thời gian dự đoán trung bình (20 lần lặp) để so sánh độ trễ inference.

In [ ]:
def avg_predict_time(predict_fn, repeats=20):
    times = []
    for _ in range(repeats):
        t0 = time.time()
        predict_fn()
        times.append(time.time() - t0)
    return float(np.mean(times))


X_acb_lstm_test, _ = prepare_lstm_sequences(
    np.vstack([X_acb_train_s[-(N_STEPS - 1) :], X_acb_test_s]),
    np.concatenate([y_acb_train.values[-(N_STEPS - 1) :], y_acb_test.values]),
    N_STEPS,
)

latency_acb = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'LSTM'],
    'Avg Predict Time (s)': [
        avg_predict_time(lambda: lr_acb.predict(X_acb_test_s)),
        avg_predict_time(lambda: rf_acb.predict(X_acb_test)),
        avg_predict_time(lambda: acb_lstm.predict(X_acb_lstm_test, verbose=0)),
    ],
    'Train Time (s)': [np.nan, np.nan, acb_lstm_metrics['Train Time (s)']],
    'Predict Time (s)': [np.nan, np.nan, acb_lstm_metrics['Predict Time (s)']],
}).set_index('Model')

display(latency_acb.round(6))


## 10. Ablation Study

Thử nghiệm số lag (3 vs 5 vs 7) với Linear Regression trên ACB.

In [ ]:
ablation_results = []
for n_lags in [3, 5, 7]:
    lag_df = create_lag_features(acb_pre, 'close', n_lags=n_lags)
    X = lag_df[[c for c in lag_df.columns if c.startswith('lag_')]]
    y = lag_df['target']
    X_tr, X_te, y_tr, y_te = time_series_split(X, y, TEST_SIZE)
    X_tr_s, X_te_s, _ = scale_features(X_tr, X_te)
    _, pred, m = run_lr(X_tr_s, X_te_s, y_tr, y_te)
    ablation_results.append({'n_lags': n_lags, **m})

ablation_df = pd.DataFrame(ablation_results).set_index('n_lags')
display(ablation_df.round(4))


## 11. Storage Evaluation

Kích thước file khi serialize các mô hình (KB).

In [ ]:
with TemporaryDirectory() as tmpdir:
    tmp = Path(tmpdir)
    paths = {
        'Linear Regression': tmp / 'lr.pkl',
        'Random Forest': tmp / 'rf.pkl',
        'LSTM': tmp / 'lstm.keras',
    }
    joblib.dump(lr_acb, paths['Linear Regression'])
    joblib.dump(rf_acb, paths['Random Forest'])
    acb_lstm.save(paths['LSTM'])

    storage_df = pd.DataFrame(
        {'Size (KB)': [p.stat().st_size / 1024 for p in paths.values()]},
        index=list(paths.keys()),
    )

display(storage_df.round(2))


## 12. Conclusion

| Khía cạnh | Nhận xét |
|-----------|----------|
| **Linear Regression** | Baseline nhanh, dễ giải thích; phù hợp quan hệ tuyến tính giữa các lag. |
| **Random Forest** | Bắt phi tuyến tốt hơn LR mà vẫn inference nhanh. |
| **ARIMA** | Mô hình chuỗi thuần túy; chất lượng phụ thuộc (p,d,q) và độ dài chuỗi. |
| **LSTM** | Học pattern phức tạp nhưng tốn thời gian train và dung lượng lưu trữ lớn hơn. |

Kết quả cụ thể (RMSE, MAE, R²) cho từng dataset nằm ở **Section 8**. Trên dữ liệu tài chính và môi trường, không có mô hình nào luôn thắng tuyệt đối — cần chọn theo độ chính xác, độ trễ và chi phí triển khai.